In [18]:

import os

# 1. Criar pastas para melhorar a organização do mini projeto

pasta_do_notebook = os.getcwd()
if os.path.basename(pasta_do_notebook) == 'notebooks':
    raiz_projeto = os.path.dirname(pasta_do_notebook)
else:
    raiz_projeto = pasta_do_notebook

caminho_data = os.path.join(raiz_projeto, 'data')
caminho_notebooks = os.path.join(raiz_projeto, 'notebooks')
caminho_output = os.path.join(raiz_projeto, 'output')

os.makedirs(caminho_data, exist_ok=True)
os.makedirs(caminho_notebooks, exist_ok=True)
os.makedirs(caminho_output, exist_ok=True)

print("Pastas verificadas e organizadas corretamente na raiz do projeto!")


Pastas verificadas e organizadas corretamente na raiz do projeto!


In [19]:
# SPRINT 1 (Configuração e ingestão)

import pandas as pd
from pathlib import Path

# 1. Executando o arquivo disponibilizado

caminho_base = Path.cwd()
arquivo = caminho_base / "data" / "halloween_vendas.csv"
candidatos = [
    caminho_base / "data" / "halloween_vendas.csv",
    caminho_base.parent / "data" / "halloween_vendas.csv",
]
arquivo = next((caminho for caminho in candidatos if caminho.is_file()), None)

if arquivo is None:
    raise FileNotFoundError(
        "Não encontrei halloween_vendas.csv em: "
        + ", ".join(str(caminho) for caminho in candidatos)
    )

df_inicial = pd.read_csv(arquivo, sep=",", encoding="utf-8")

# 2. Qauntidade de linhas no arquivo
print(f"Total de registros encontrados: {len(df_inicial)} linhas.")

# 3. Significado das colunas principais

print("\npedido_id: Número do pedido.")
print("categoria e canal_venda: O tipo do produto e onde ele foi comprado.")
print("quantidade e preco_unitario: Quantos itens foram comprados e o valor de cada um.")
print("data_venda e data_entrega: Quando o cliente comprou e quando o produto foi entregue.")

erros_qtd = (df_inicial['quantidade'].isna() | (df_inicial['quantidade'] <= 0)).sum()
erros_preco = (df_inicial['preco_unitario'].isna() | (df_inicial['preco_unitario'] < 0)).sum()

# 4.Calculando o total de pedidos afetados sem somar ocorrências sobrepostas (usando ID único)
linhas_com_erro = df_inicial[
    (df_inicial['quantidade'].isna()) | (df_inicial['quantidade'] <= 0) |
    (df_inicial['preco_unitario'].isna()) | (df_inicial['preco_unitario'] < 0)
]
total_pedidos_afetados = linhas_com_erro['pedido_id'].nunique()

# 5. Exibindo o contador solicitado pelo enunciado
print("\nQuantidade de erros")
print(f"Resultado com erro em 'quantidade': {erros_qtd}")
print(f"Resultado com erro em 'preco_unitario': {erros_preco}")
print(f"Total de pedidos: {total_pedidos_afetados}\n")

# 6. Mostrando 3 exemplos conforme o enunciado

print("Mostrando os 3 primeiros itens do arquivo")
display(df_inicial.head(3))

Total de registros encontrados: 1500 linhas.

pedido_id: Número do pedido.
categoria e canal_venda: O tipo do produto e onde ele foi comprado.
quantidade e preco_unitario: Quantos itens foram comprados e o valor de cada um.
data_venda e data_entrega: Quando o cliente comprou e quando o produto foi entregue.

Quantidade de erros
Resultado com erro em 'quantidade': 7
Resultado com erro em 'preco_unitario': 12
Total de pedidos: 19

Mostrando os 3 primeiros itens do arquivo


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110.0,57.00,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


In [20]:
# SPRINT 2 (Transformação de Strings)

import pandas as pd
import re

# 1. Criar uma cópia do df_inicial para não alterar os dados originais
df_sprint2 = df_inicial.copy()

# 2. Resultado do ANTES
print("\n   ANTES")
print("Categorias únicas encontradas:", df_sprint2['categoria'].unique()[:5].tolist())
print("Canais únicos encontrados:", df_sprint2['canal_venda'].unique().tolist())

# 3. Processamento de dados usando RE.SUB

cat_vazias_preenchidas = 0
cat_nao_vazias_alteradas = 0
categorias_novas = []

for valor in df_sprint2['categoria']:
    # 3.1. Verificar se está vazio ou nulo
    if pd.isna(valor) or str(valor).strip() == "":
        cat_vazias_preenchidas += 1
        categorias_novas.append("sem categoria")
    else:
        texto_original = str(valor)
        # 3.2. Deixar em minúsculo e remove espaços no inicio/final
        texto_tratado = texto_original.strip().lower()

        # 3.3 re.sub para manter apenas letras de 'a' a 'z' e espaços
        texto_limpo = re.sub(r'[^a-z\s]', '', texto_tratado)
        texto_limpo = texto_limpo.strip()

        # 3.4 Se após a limpeza o texto sumiu, vira sem categoria
        if texto_limpo == "":
            cat_vazias_preenchidas += 1
            categorias_novas.append("sem categoria")
        else:
            # Remover ('!'), se for diferente do original
            if texto_limpo != texto_original:
                cat_nao_vazias_alteradas += 1
            categorias_novas.append(texto_limpo)

df_sprint2['categoria'] = categorias_novas

# 4. Padronização nas descrições

canais_novos = []
total_canais_padronizados = 0

for canal in df_sprint2['canal_venda']:
    if pd.isna(canal):
        canais_novos.append("Não Reconhecido")
    else:
        texto_canal = str(canal).strip().lower()
        total_canais_padronizados += 1

        if 'loja' in texto_canal or 'física' in texto_canal or 'fisica' in texto_canal:
            canais_novos.append("Loja física")
        elif 'web' in texto_canal:
            canais_novos.append("Web")
        elif 'mercado' in texto_canal:
            canais_novos.append("Mercado")
        else:
            canais_novos.append("Não Reconhecido")

# 5. Resultado do DEPOIS

df_sprint2['canal_venda'] = canais_novos

print("\n   DEPOIS ")
print("Categorias tratadas:", df_sprint2['categoria'].unique().tolist())
print("Canais padronizados:", df_sprint2['canal_venda'].unique().tolist())

print("\n   Resultados dos erros encontrados")
print(f"Categorias vazias que foram preenchidas: {cat_vazias_preenchidas}")
print(f"Categorias não vazias que foram alteradas/limpas: {cat_nao_vazias_alteradas}")
print(f"Total de canais de venda padronizados: {total_canais_padronizados}")


   ANTES
Categorias únicas encontradas: ['Fantasia', '  FANTASIA!!!  ', 'Acessorio', ' Fantasia ', '  ACESSORIO!!!  ']
Canais únicos encontrados: ['Loja física', 'Web', 'Mercado', ' loja física ', ' mercado ', ' web ']

   DEPOIS 
Categorias tratadas: ['fantasia', 'acessorio', 'sem categoria']
Canais padronizados: ['Loja física', 'Web', 'Mercado']

   Resultados dos erros encontrados
Categorias vazias que foram preenchidas: 18
Categorias não vazias que foram alteradas/limpas: 1482
Total de canais de venda padronizados: 1500


In [21]:
# SPRINT 3 (Nulos e Validação)

import pandas as pd

# 1. Cria uma cópia do df_sprint2 para continuar o projeto sem alterar o anterior
df_sprint3 = df_sprint2.copy()

# 2. Datas sem entregas e pedidos cancelados

pedidos_sem_entrega_cancelados = []
pedidos_sem_entrega_outros_status = []

#3. Analisando linha por linha
for indice, linha in df_sprint3.iterrows():
    #3.1. Verificar se a data de entrega está vazia
    if pd.isna(linha['data_entrega']) or str(linha['data_entrega']).strip() == "":

        # 3.2. Se estiver vazia e o status for Cancelado, vai para o Resultado 1
        if linha['status_pedido'] == 'Cancelado':
            pedidos_sem_entrega_cancelados.append(linha)
        # 3.3. Se estiver vazia, mas tiver OUTRO STATUS, vai para o Resultado 2
        else:
            pedidos_sem_entrega_outros_status.append(linha)

# 4. Converter as listas de volta para DataFrames para poder contar e exibir
df_grupo1 = pd.DataFrame(pedidos_sem_entrega_cancelados)
df_grupo2 = pd.DataFrame(pedidos_sem_entrega_outros_status)

# 5. Resultados

print("\n   Resultados da Validação de Entregas")
print(f"Pedidos sem entrega e CANCELADOS - Resultado 1: {len(df_grupo1)}")
print(f"Pedidos com data vazia/OUTROS STATUS - Resultado 2: {len(df_grupo2)}")

print("\n   Exemplos de pedidos sem entrega com outros status:")
# 5.1. Se houver algum pedido no Resultado 2, mostrar até 3 exemplos e concluir que a hipótese é falsa
if len(df_grupo2) > 0:
    display(df_grupo2[['pedido_id', 'status_pedido', 'data_entrega']].head(3))
    print("\nExistem pedidos sem entrega que não foram cancelados. A hipótese é FALSA")
else:
    print("Nenhum exemplo encontrado. A hipótese é VERDADEIRA.")



   Resultados da Validação de Entregas
Pedidos sem entrega e CANCELADOS - Resultado 1: 109
Pedidos com data vazia/OUTROS STATUS - Resultado 2: 34

   Exemplos de pedidos sem entrega com outros status:


,pedido_id,status_pedido,data_entrega
49,1170,Concluído,NaN
117,990,Concluído,NaN
152,740,Concluído,NaN



Existem pedidos sem entrega que não foram cancelados. A hipótese é FALSA


In [22]:
# SPRINT 4 (Regras de Negócio e Datas)
import pandas as pd
from datetime import datetime

# 1. Criar uma cópia do df_sprint3 para trabalhar com as datas
df_sprint4 = df_sprint3.copy()

# 2. Função simples para tratar e classificar as datas
def processar_data_enunciado(texto_data):
    if pd.isna(texto_data) or str(texto_data).strip() == "" or str(texto_data) == "data_invalida":
        return None, None, None, "inválida/vazia"
    try:
        # 2.1. Lendo o formato original do arquivo (Ano-Mês-Dia Hora:Minuto:Segundo)
        dt_objeto = datetime.strptime(str(texto_data).strip(), "%Y-%m-%d %H:%M:%S")

        # 2.2. Extraindo as informações solicitadas no enunciado
        ano_mes_texto = dt_objeto.strftime("%Y-%m") # Agrupamento correto por Ano-Mês
        data_br = dt_objeto.strftime("%d/%m/%Y")    # Apresentação brasileira (dia/mês/ano)

        return dt_objeto.year, ano_mes_texto, data_br, "válida"
    except Exception:
        return None, None, None, "inválida/vazia"

# 3. Criar as colunas vazias
df_sprint4['ano'] = None
df_sprint4['ano_mes'] = None
df_sprint4['data_venda_br'] = None
df_sprint4['data_venda_status'] = None

# 4. Aplicar função linha por linha
for indice, linha in df_sprint4.iterrows():
    res = processar_data_enunciado(linha['data_venda'])
    df_sprint4.at[indice, 'ano'] = res[0]
    df_sprint4.at[indice, 'ano_mes'] = res[1]
    df_sprint4.at[indice, 'data_venda_br'] = res[2]
    df_sprint4.at[indice, 'data_venda_status'] = res[3]

# 5. Contagem de datas inválidas encontradas
total_datas_invalidas = (df_sprint4['data_venda_status'] == "inválida/vazia").sum()

print("\n   Resultado do Processamento de Datas")
print(f"Datas de venda inválidas ou vazias identificadas: {total_datas_invalidas}")


   Resultado do Processamento de Datas
Datas de venda inválidas ou vazias identificadas: 4


In [23]:
# SPRINT 5 (Relatório de Status Manual e Exploração)

import os
from pathlib import Path

# 1. Criar cópia a partir da sprint anterior para gerar o relatório final

df_final = df_sprint4.copy()
total_linhas = len(df_final)

# 2. Calcular os números para o relatório de status
categorias_informadas = (df_final['categoria'] != 'sem categoria').sum()
total_cancelados = (df_final['status_pedido'] == 'Cancelado').sum()

# 3. Exibir os resultados do SPRINT 3
grupo_cancelados = len(df_final[(df_final['status_pedido'] == 'Cancelado')])
grupo_outros_status = len(df_final[(df_final['data_venda_status'] == 'inválida/vazia')])

erros_qtd = erros_qtd if 'erros_qtd' in locals() else 0
erros_preco = erros_preco if 'erros_preco' in locals() else 0

# 4. Exibir relatório de status

print("\n   RELATÓRIO DE STATUS")
print(f"Total de linhas lidas: {total_linhas}")
print(f"Categorias preenchidas/padronizadas: {categorias_informadas}")
print(f"Inconsistências numéricas: Quantidade ({erros_qtd}), Preço ({erros_preco})")
print(f"Datas de venda inválidas: {total_datas_invalidas}")
print(f"Total de pedidos cancelados: {total_cancelados}")

print(f"\nRESULTADOS 1 - Pedidos sem entrega e CANCELADOS): {grupo_cancelados} pedidos.")
print(f"RESULTADOS 2 - Pedidos com data vazia/OUTROS STATUS): {grupo_outros_status} pedidos.")

# 4. Quantidades

print("\nQuantidade de pedidos por canais de vendas")
# 4.1 Filtra apenas canais reconhecidos
df_canal_valido = df_final[df_final['canal_venda'] != 'Não Reconhecido']
print(df_canal_valido['canal_venda'].value_counts().to_string())
print(f"Exclusões nesta contagem: {total_linhas - len(df_canal_valido)} registros sem canal válido.")

print("\nQuantidade de pedidos Ano-Mês")
# 4.2 Filtrar apenas datas de venda válidas
df_data_valida = df_final[df_final['data_venda_status'] == 'válida']
print(df_data_valida['ano_mes'].value_counts().sort_index().to_string())
print(f"Exclusões nesta contagem: {total_datas_invalidas} registros com datas inválidas.")

# 5. Resultados finais

print("\nRESULTADO 1 - Canais de vendas")
print("O canal de vendas que mais foi utilizado em Outubro foi 'web' em pedidos efetuados.")
print("A internet é o canal preferido dos clientes na temporada de Halloween.")
print("Conta apenas pedidos criados. Não reflete o faturamento real, pois nem todos os pedidos foram efetuados.")

print("\nRESULTADO 2 - Período")
print("O mês de Outubro de 2025 teve o maior período de vendas, com 410 pedidos.")
print("As vendas ficam concentradas na época exata do Halloween 31 de Outubro.")
print("Mostra o volume de pedidos registrados, mas não garante que todos foram entregues no prazo estipulado.")

# 6. Salvar arquivo de analise final na pasta /output)

caminho_final_csv = os.path.join(caminho_output, 'pedidos_halloween_finais.csv')
df_final.to_csv(caminho_final_csv, index=False)



   RELATÓRIO DE STATUS
Total de linhas lidas: 1500
Categorias preenchidas/padronizadas: 1482
Inconsistências numéricas: Quantidade (7), Preço (12)
Datas de venda inválidas: 4
Total de pedidos cancelados: 109

RESULTADOS 1 - Pedidos sem entrega e CANCELADOS): 109 pedidos.
RESULTADOS 2 - Pedidos com data vazia/OUTROS STATUS): 4 pedidos.

Quantidade de pedidos por canais de vendas
canal_venda
Web            666
Loja física    551
Mercado        283
Exclusões nesta contagem: 0 registros sem canal válido.

Quantidade de pedidos Ano-Mês
ano_mes
2024-08     77
2024-09    104
2024-10    393
2024-11    174
2025-08     62
2025-09     94
2025-10    410
2025-11    182
Exclusões nesta contagem: 4 registros com datas inválidas.

RESULTADO 1 - Canais de vendas
O canal de vendas que mais foi utilizado em Outubro foi 'web' em pedidos efetuados.
A internet é o canal preferido dos clientes na temporada de Halloween.
Conta apenas pedidos criados. Não reflete o faturamento real, pois nem todos os pedidos 

In [24]:
with open("README.md", "w", encoding="utf-8") as f:
    f.write("""# Mini projeto de vendas de Halloween
Este mini projeto foi feito para limpar, organizar e analisar uma tabela de vendas de Halloween usando python e pandas.
## Criei pastas no projeto, para melhor organização
* `data/`: onde fica o arquivo bruto original (`halloween_vendas.csv`).
* `notebooks/`: onde fica o código que criei (`analise_final_halloween.ipynb`).
* `output/`: onde salvei o resultado final(`pedidos_halloween_finais.csv`).
##Regras que usei e justificativas
###1. Sprint 1 - Configuração e Ingestão
* Quantidade: Deve ser maior que 0.
* Preço: Deve ser preenchido e não pode ser negativo.
* Realizado: Se a linha tiver erro, apenas contar o problema em vez de inventar colocar um valor qualquer no lugar.
###2. Sprint 2 - Transformação de Strings
* Categoria: Se estava vazia, escrever "sem categoria". Usei o comando re.sub para apagar os pontos de exclamação (`!`) e deixar tudo em letra minúscula.
* Canal de Venda: Apenas três nomes padrão foram identificados, sem acréscimos de acentos ou espaçamento: `Web`, `Loja física` e `Mercado`.
###3. Sprint 3 - Nulos e Validação
* Após análise dos dados percebi que todo pedido sem data de entrega estava Cancelado".
* Conclui que esta informação é FALSO, porque existem vários pedidos com status "Concluído" que também não têm a data de entrega preenchida.
###4. Sprint 4 - Regras de Negócio e Datas
* Formato: Realizei a conversão das datas do arquivo original para o formato do Brasil (dia/mês/ano).
* Se a data estava invalida ou vazia, apenas pulei para continuar o código.
""")

with open("report.md", "w", encoding="utf-8") as f:
    f.write("""#  Relatório final das vendas
## Números da Qualidade dos Dados
Depois de realizar a analise das 1500 linhas do arquivo, foram encontradas as seguintes informações:
* **Categorias atualizadas:** 1482 linhas (18 estavam vazias e viraram "sem categoria").
* **Erros de Quantidade:** 7 linhas com problemas.
* **Erros de Preço:** 12 linhas com problemas.
* **Total de pedidos com algum erro numérico:** 19 pedidos no total.
* **Datas de venda com erro:** 4 datas inválidas.
* **Total de pedidos Cancelados:** 109 pedidos.
### Pedidos com divergência
* **Pedidos sem entrega que foram **Cancelados**: 109 pedidos.
* **Pedidos sem entrega que estão com **Outros Status** ou **Vazia** sem informações: 34 pedidos.
* **Conclusão: A hipótese é **FALSA**. Há 34 pedidos que deram como ‘Concluídos’, mas estão com divergência na data de entrega.

## Descobertas com os dados apurados
*Importante: Essas contagens mostram apenas os pedidos que foram criados no sistema, não significa realmente faturou.*

### 1. Analise em relação aos canais de vendas
* **Web:** 666 pedidos.
* **Loja física:** 551 pedidos.
* **Mercado:** 283 pedidos.
***Resultado:** O canal **Web** é o mais usado para realização das vendas.
***Porém:** Esse número conta até os pedidos que foram cancelados depois, então não reflete o lucro real.
""")